In [3]:
!wget -q -O google-chrome-stable_current_amd64.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get update
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get -fy install
!pip install -U selenium pandas beautifulsoup4

'wget'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
'apt-get'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
'dpkg'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
'apt-get'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


  Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl (9.7 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas None


error: uninstall-no-record-file

× Cannot uninstall pandas None
╰─> The package's contents are unknown: no RECORD file was found for pandas.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps pandas==2.3.3


In [ ]:
# =====================================================================
# K리그 선수 데이터 크롤러
# ★ Transfermarkt K리그 시즌 표기 규칙:
#    실제 2025시즌 → saison_id=2024 (1년 전 숫자 사용)
#    실제 2026시즌 → saison_id=2025
# =====================================================================
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# ★ 수집할 실제 시즌 (표기용)
REAL_SEASON = 2025
# ★ Transfermarkt에서 사용하는 saison_id (실제시즌 - 1)
SAISON_ID = REAL_SEASON - 2  # 2024

columns = [
    'Name in home country', 'Date of birth/Age', 'Place of birth',
    'Height', 'Citizenship', 'Position', 'Player agent',
    'Current club', 'Joined', 'Contract expires', 'Value'
]

# ★ 리그 메인 페이지 — saison_id=2024 로 2025시즌 구단 목록 수집
league_urls = [
    f'https://www.transfermarkt.com/k-league-1/startseite/wettbewerb/RSK1/plus/?saison_id={SAISON_ID}',
    f'https://www.transfermarkt.com/k-league-2/startseite/wettbewerb/RSK2/plus/?saison_id={SAISON_ID}'
]

print(f"1단계: K리그 1, 2 구단(Club) URL을 수집합니다... (실제 시즌: {REAL_SEASON} / saison_id: {SAISON_ID})")
club_ids = {}  # {verein_id: team_name_slug}

for l_url in league_urls:
    res = requests.get(l_url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')

    club_table = soup.find('table', class_='items')
    if club_table:
        tags = club_table.select('td.hauptlink.no-border-links a')
        for tag in tags:
            href = tag.get('href')
            if href and '/startseite/verein/' in href:
                # verein ID 추출 (예: /fc-seoul/startseite/verein/6500/... → 6500)
                m = re.search(r'/startseite/verein/(\d+)', href)
                slug = href.split('/')[1]  # 팀 이름 슬러그
                if m:
                    club_ids[m.group(1)] = slug

    time.sleep(1)

# ★ 핵심: 유저가 발견한 URL 형식으로 구단 스쿼드 페이지 구성
# https://www.transfermarkt.com/{slug}/kader/verein/{id}/plus/0/galerie/0?saison_id={SAISON_ID}
club_urls = [
    f'https://www.transfermarkt.com/{slug}/kader/verein/{vid}/plus/0/galerie/0?saison_id={SAISON_ID}'
    for vid, slug in club_ids.items()
]

print(f"-> 총 {len(club_urls)}개의 구단 URL 수집 완료!\n")

print("================ [수집된 구단 리스트 확인] ================")
for idx, c_url in enumerate(club_urls, 1):
    team_name = c_url.split('/')[3].replace('-', ' ').title()
    print(f"{idx:02d}. {team_name}\n    ({c_url})")
print("===========================================================\n")


print("2단계: 각 구단 스쿼드 페이지에서 선수(Player) URL을 수집합니다...")
player_urls = set()

for c_url in club_urls:
    try:
        res = requests.get(c_url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        tags = soup.select('table.items td.hauptlink a')
        added = 0
        for tag in tags:
            href = tag.get('href')
            if href and '/profil/spieler/' in href:
                player_urls.add('https://www.transfermarkt.com' + href)
                added += 1

        team_name = c_url.split('/')[3].replace('-', ' ').title()
        print(f"  {team_name}: {added}명 수집")
        time.sleep(random.uniform(1, 2))
    except Exception as e:
        print(f"구단 정보 수집 에러 ({c_url}): {e}")

player_urls = list(player_urls)
print(f"\n-> 총 {len(player_urls)}명의 선수 URL 수집 완료!\n")


print("3단계: 각 선수의 상세 프로필 정보를 수집합니다... (시간이 다소 소요됩니다)")
player_data = []

# 테스트 시 상위 5명만 보려면 아래 주석 해제
# player_urls = player_urls[:5]

count = 1
total_players = len(player_urls)

for url in player_urls:
    try:
        if count % 10 == 0:
            print(f"수집 진행 중... ({count}/{total_players})")

        res = requests.get(url, headers=headers)
        res.raise_for_status()
        soup = BeautifulSoup(res.content, 'html.parser')

        info_dict = {col: '-' for col in columns}

        # 1. 상세 정보 테이블 파싱
        info_box = soup.find('div', class_='info-table info-table--right-space')
        if info_box:
            rows = info_box.find_all('span', class_='info-table__content')
            for i in range(0, len(rows) - 1, 2):
                label = rows[i].text.strip().replace(':', '')
                value = rows[i+1].text.strip()
                if label in info_dict:
                    info_dict[label] = value

        # 2. 시장 가치(Value) 파싱 + Last update 즉시 제거
        val_tag = soup.find('a', class_='data-header__market-value-wrapper')
        if val_tag:
            raw_val = val_tag.text.strip().split('\n')[0]
            raw_val = raw_val.split(' Last update')[0].strip()
            info_dict['Value'] = raw_val

        # 3. 이름 보완 (외국인 선수 등)
        if info_dict['Name in home country'] == '-':
            header = soup.find('h1', class_='data-header__headline-wrapper')
            if header:
                raw_name = header.get_text(separator=" ", strip=True)
                info_dict['Name in home country'] = re.sub(r'^#\d+\s+', '', raw_name)

        player_data.append([info_dict[col] for col in columns])
        time.sleep(random.uniform(1.0, 2.0))
        count += 1

    except Exception as e:
        print(f"선수 상세 수집 오류 ({url}): {e}")
        continue

# 4. 데이터프레임 생성 및 CSV 저장
df = pd.DataFrame(player_data, columns=columns)

print("\n[수집된 데이터 확인]")
print(df.head(10))

file_name = f'Transfermarkt_KLeague_Players_{REAL_SEASON}.csv'
df.to_csv(file_name, index=False, encoding='utf-8-sig')
print(f"\n'{file_name}' 저장 완료! (총 {len(df)}명 수집)")

1단계: K리그 1, 2 구단(Club) URL을 수집합니다... (실제 시즌: 2025 / saison_id: 2023)
